# 03 - Feature Extraction

Mengubah teks menjadi fitur numerik menggunakan TF-IDF. Data di-split terlebih dahulu sebelum TF-IDF fit agar tidak terjadi data leakage (IDF scores dari test set bocor ke training set).

In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

## 1. Load Data

In [2]:
df = pd.read_csv('../data/processed/cleaned_data.csv')
print(f'Baris: {len(df)}')
print(f'Label:\n{df["label"].value_counts()}')
print(f'\nContoh teks bersih:\n{df["clean_text"].iloc[0][:200]}')

Baris: 44689
Label:
label
0    23478
1    21211
Name: count, dtype: int64

Contoh teks bersih:
donald trump sends out embarrassing new year’s eve message this is disturbing donald trump just couldn t wish all americans a happy new year and leave it at that instead he had to give a shout out to 


## 2. Split Data

Split dilakukan SEBELUM TF-IDF fit. Ini penting agar IDF scores hanya dihitung dari training data, bukan dari seluruh corpus. Jika tidak, model "melihat" data test saat fitur dibuat, sehingga evaluasi menjadi terlalu optimis.

In [3]:
df['clean_text'] = df['clean_text'].fillna('')

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_text'], df['label'],
    test_size=0.2, stratify=df['label'], random_state=43
)

print(f'Training: {len(X_train_text)} teks')
print(f'Testing : {len(X_test_text)} teks')
print(f'\nDistribusi train: {np.bincount(y_train)}')
print(f'Distribusi test : {np.bincount(y_test)}')

Training: 35751 teks
Testing : 8938 teks

Distribusi train: [18782 16969]
Distribusi test : [4696 4242]


## 3. TF-IDF Vectorizer

TF-IDF hanya di-fit pada data training. Data test ditransform menggunakan vectorizer yang sama, tapi tanpa fit ulang. Parameter: ngram_range=(1,2) untuk unigram + bigram, max_features=50000, sublinear_tf=True untuk scaling log.

In [4]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000,
    sublinear_tf=True
)

X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

print(f'Matriks training: {X_train.shape}')
print(f'Matriks testing : {X_test.shape}')

Matriks training: (35751, 50000)
Matriks testing : (8938, 50000)


## 4. Top Fitur per Kelas

Melihat fitur TF-IDF dengan skor tertinggi di masing-masing kelas berdasarkan data training saja.

In [5]:
feature_names = tfidf.get_feature_names_out()
y_train_arr = y_train.values

mean_tfidf_fake = X_train[y_train_arr == 0].mean(axis=0).A1
mean_tfidf_true = X_train[y_train_arr == 1].mean(axis=0).A1

top_fake_idx = mean_tfidf_fake.argsort()[-20:][::-1]
top_true_idx = mean_tfidf_true.argsort()[-20:][::-1]

print('Top 20 fitur - FAKE NEWS:')
for idx in top_fake_idx:
    print(f'  {feature_names[idx]:30s} {mean_tfidf_fake[idx]:.4f}')

print('\nTop 20 fitur - TRUE NEWS:')
for idx in top_true_idx:
    print(f'  {feature_names[idx]:30s} {mean_tfidf_true[idx]:.4f}')

Top 20 fitur - FAKE NEWS:
  the                            0.0365
  to                             0.0311
  of                             0.0285
  and                            0.0281
  in                             0.0248
  that                           0.0243
  trump                          0.0236
  is                             0.0235
  for                            0.0211
  it                             0.0200
  he                             0.0193
  this                           0.0191
  you                            0.0188
  on                             0.0188
  video                          0.0187
  his                            0.0180
  with                           0.0179
  was                            0.0172
  they                           0.0161
  are                            0.0161

Top 20 fitur - TRUE NEWS:
  the                            0.0393
  to                             0.0318
  of                             0.0294
  in                       

Top fitur masih didominasi kata-kata umum. Perbedaan mulai terlihat di peringkat bawah: fake news punya "video", "you", "his" yang bersifat personal dan emosional, sementara true news punya "said", "us", "by" yang lebih formal dan jurnalistik.

## 5. Simpan Hasil

In [6]:
import scipy.sparse as sp

sp.save_npz('../data/processed/X_train_tfidf.npz', X_train)
sp.save_npz('../data/processed/X_test_tfidf.npz', X_test)
np.save('../data/processed/y_train.npy', y_train.values)
np.save('../data/processed/y_test.npy', y_test.values)
joblib.dump(tfidf, '../models/tfidf_vectorizer.pkl')

print(f'X_train_tfidf.npz : {X_train.shape}')
print(f'X_test_tfidf.npz  : {X_test.shape}')
print(f'y_train.npy       : {y_train.shape}')
print(f'y_test.npy        : {y_test.shape}')
print(f'tfidf_vectorizer.pkl : {tfidf}')
print('\nSemua file tersimpan.')

X_train_tfidf.npz : (35751, 50000)
X_test_tfidf.npz  : (8938, 50000)
y_train.npy       : (35751,)
y_test.npy        : (8938,)
tfidf_vectorizer.pkl : TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)

Semua file tersimpan.
